# DWS Water Quality PipelineEnd-to-end pipeline: scrape DWS stations → download data → match to train/test → CatBoost modeling → hybrid submission.**Output**: `submission_dws_hybrid.csv`

## Setup

In [ ]:
import os
import re
import io
import time
import glob
import zipfile
import warnings

import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from scipy.spatial.distance import cdist
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans
from catboost import CatBoostRegressor
from tqdm import tqdm

warnings.filterwarnings('ignore')

BASE_DIR = './data'
ZIP_DIR = os.path.join(BASE_DIR, 'zips')
CSV_DIR = os.path.join(BASE_DIR, 'csvs')
os.makedirs(ZIP_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

TARGETS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

## Part 1: Scrape DWS Station ListsScrape station lists from all drainage regions (A–X) on the DWS website.

In [ ]:
def scrape_region(region_letter):
    """Scrape a DWS region page to extract surface water station metadata."""
    url = f"https://www.dws.gov.za/iwqs/wms/data/{region_letter}_reg_WMS_nobor.htm"
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
    except Exception:
        return []

    soup = BeautifulSoup(resp.content, 'html.parser')
    stations = []

    for table in soup.find_all('table'):
        for row in table.find_all('tr'):
            cells_ = row.find_all('td')
            if len(cells_) < 10:
                continue
            try:
                station_id_elem = cells_[0].find('b') or cells_[0].find('strong')
                if not station_id_elem:
                    continue
                station_id = station_id_elem.get_text(strip=True)
                if not station_id or not re.match(r'[A-Z]', station_id):
                    continue

                data_link = None
                for cell in cells_[1:3]:
                    a_tag = cell.find('a')
                    if a_tag and a_tag.get('href', '').endswith('.zip'):
                        data_link = a_tag['href']
                        break
                if data_link is None:
                    continue

                lat = float(cells_[-2].get_text(strip=True))
                lon = float(cells_[-1].get_text(strip=True))

                desc = cells_[3].get_text(strip=True) if len(cells_) > 3 else ''
                n_samples = 0
                try:
                    n_samples = int(cells_[5].get_text(strip=True)) if len(cells_) > 5 else 0
                except ValueError:
                    pass
                first_date = cells_[6].get_text(strip=True) if len(cells_) > 6 else ''
                last_date = cells_[7].get_text(strip=True) if len(cells_) > 7 else ''

                full_url = data_link if data_link.startswith('http') else \
                    f"https://www.dws.gov.za/iwqs/wms/data/{data_link}"

                stations.append({
                    'station_id': station_id.replace(' ', '_'),
                    'region': region_letter,
                    'description': desc,
                    'latitude': lat, 'longitude': lon,
                    'n_samples': n_samples,
                    'first_date': first_date, 'last_date': last_date,
                    'zip_url': full_url,
                })
            except Exception:
                continue
    return stations


ALL_REGIONS = 'A B C D E F G H J K L M N P Q R S T U V W X'.split()
all_stations = []

for region in ALL_REGIONS:
    stn = scrape_region(region)
    all_stations.extend(stn)
    print(f"Region {region}: {len(stn)} stations")
    time.sleep(0.5)

stations_df = pd.DataFrame(all_stations)
stations_df.to_csv(os.path.join(BASE_DIR, 'dws_stations_all.csv'), index=False)
print(f"\nTotal stations scraped: {len(stations_df)}")

## Part 2: Download Station Data ZIPsDownloads each station's ZIP file. Skips already-downloaded files for resumability.

In [ ]:
def download_zip(url, save_dir, station_id):
    """Download a station ZIP file; skip if already on disk."""
    filepath = os.path.join(save_dir, f"{station_id}.zip")
    if os.path.exists(filepath):
        return filepath, 'skipped'
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        with open(filepath, 'wb') as f:
            f.write(resp.content)
        return filepath, 'downloaded'
    except Exception as e:
        return None, f'error: {e}'


results = {'downloaded': 0, 'skipped': 0, 'error': 0}

for _, row in tqdm(stations_df.iterrows(), total=len(stations_df), desc='Downloading ZIPs'):
    _, status = download_zip(row['zip_url'], ZIP_DIR, row['station_id'])
    if status == 'downloaded':
        results['downloaded'] += 1
        time.sleep(0.3)
    elif status == 'skipped':
        results['skipped'] += 1
    else:
        results['error'] += 1

print(f"Download results: {results}")

## Part 3: Extract ZIPs & Build Unified Database

In [ ]:
DWS_RAW_COLS = {
    'date_time': 'date_time',
    'TAL_Diss_Water': 'TAL',
    'EC_Phys_Water': 'EC',
    'PO4_P_Diss_Water': 'PO4_P',
    'pH_Diss_Water': 'pH',
    'Ca_Diss_Water': 'Ca',
    'Mg_Diss_Water': 'Mg',
    'Na_Diss_Water': 'Na',
    'Cl_Diss_Water': 'Cl',
    'SO4_Diss_Water': 'SO4',
    'P_Tot_Water': 'P_Tot',
    'Station': 'Station',
}

DWS_NUMERIC_COLS = ['TAL', 'EC', 'PO4_P', 'pH', 'Ca', 'Mg', 'Na', 'Cl', 'SO4', 'P_Tot']


def extract_and_parse_zip(zip_path, station_id):
    """Extract key water-quality columns from a station ZIP file."""
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            csv_files = [f for f in z.namelist()
                         if f.lower().endswith('.csv') or f.lower().endswith('.txt')]
            if not csv_files:
                return None
            with z.open(csv_files[0]) as f:
                try:
                    df = pd.read_csv(f, encoding='utf-8', na_values=['#N/A', '', 'NA', 'n/a'])
                except Exception:
                    f.seek(0)
                    df = pd.read_csv(f, encoding='latin1', na_values=['#N/A', '', 'NA', 'n/a'])
        if df.empty:
            return None

        df.columns = df.columns.str.strip()
        result = pd.DataFrame()
        result['station_id'] = station_id

        for orig_col, new_col in DWS_RAW_COLS.items():
            matched = [c for c in df.columns if c.lower() == orig_col.lower()]
            col_name = orig_col if orig_col in df.columns else (matched[0] if matched else None)
            if col_name:
                result[new_col] = df[col_name].values

        if 'date_time' not in result.columns:
            return None

        result['station_id'] = station_id
        result['date_time'] = pd.to_datetime(result['date_time'], errors='coerce')
        for col in DWS_NUMERIC_COLS:
            if col in result.columns:
                result[col] = pd.to_numeric(result[col], errors='coerce')
        return result
    except Exception:
        return None


zip_files = glob.glob(os.path.join(ZIP_DIR, '*.zip'))
all_data = []
for zp in tqdm(zip_files, desc='Parsing ZIPs'):
    sid = os.path.basename(zp).replace('.zip', '')
    parsed = extract_and_parse_zip(zp, sid)
    if parsed is not None and len(parsed) > 0:
        all_data.append(parsed)

dws_db = pd.concat(all_data, ignore_index=True)
station_coords = stations_df[['station_id', 'latitude', 'longitude']].copy()
dws_db = dws_db.merge(station_coords, on='station_id', how='left')

db_path = os.path.join(BASE_DIR, 'dws_water_quality_db.csv')
dws_db.to_csv(db_path, index=False)
print(f"Database: {len(dws_db):,} records, {dws_db['station_id'].nunique()} stations")

## Part 4: Match DWS Features to Train & Test

In [ ]:
train = pd.read_csv(os.path.join(BASE_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(BASE_DIR, 'submission_template.csv'))
stations_df = pd.read_csv(os.path.join(BASE_DIR, 'dws_stations_all.csv'))
dws_db = pd.read_csv(db_path, parse_dates=['date_time'])

DWS_FEATURE_COLS = ['TAL', 'EC', 'PO4_P', 'pH', 'Ca', 'Mg', 'Na', 'Cl', 'SO4']


def match_dws_fast(df, dws_db, stations_df):
    """For each row, find nearest DWS station and its closest-date measurement."""
    dws_coords = stations_df[['station_id', 'latitude', 'longitude']].drop_duplicates('station_id')
    df_locs = df.groupby(['Latitude', 'Longitude']).size().reset_index().rename(columns={0: 'n'})

    dist_matrix = cdist(
        df_locs[['Latitude', 'Longitude']].values,
        dws_coords[['latitude', 'longitude']].values,
    ) * 111  # approximate km

    df_locs['dws_1st'] = dws_coords['station_id'].values[dist_matrix.argmin(axis=1)]
    df_locs['dws_1st_dist'] = dist_matrix.min(axis=1)

    df = df.copy()
    df = df.merge(df_locs[['Latitude', 'Longitude', 'dws_1st', 'dws_1st_dist']],
                  on=['Latitude', 'Longitude'], how='left')
    df['Sample Date'] = pd.to_datetime(df['Sample Date'], dayfirst=True)

    results = {col: np.full(len(df), np.nan) for col in DWS_FEATURE_COLS}
    results['dws_days_diff'] = np.full(len(df), np.nan)
    results['dws_dist_km'] = df['dws_1st_dist'].values

    for station_id, group in tqdm(df.groupby('dws_1st'), desc='Matching'):
        sdata = dws_db[dws_db['station_id'] == station_id].copy()
        if len(sdata) == 0:
            continue
        sdata = sdata.sort_values('date_time')
        for idx, row in group.iterrows():
            date_diffs = (sdata['date_time'] - row['Sample Date']).abs()
            closest_idx = date_diffs.idxmin()
            results['dws_days_diff'][idx] = date_diffs[closest_idx].days
            best_row = sdata.loc[closest_idx]
            for col in DWS_FEATURE_COLS:
                if col in sdata.columns:
                    results[col][idx] = best_row.get(col, np.nan)

    for col in DWS_FEATURE_COLS:
        df[f'dws_{col}'] = results[col]
    df['dws_days_diff'] = results['dws_days_diff']
    df['dws_dist_km'] = results['dws_dist_km']
    return df


print("Matching DWS → train...")
train_matched = match_dws_fast(train, dws_db, stations_df)

print("Matching DWS → test...")
test_matched = match_dws_fast(test, dws_db, stations_df)

print(f"\nTrain DWS coverage:")
for col in DWS_FEATURE_COLS:
    print(f"  dws_{col}: {train_matched[f'dws_{col}'].notna().mean()*100:.1f}%")

## Part 5: Feature Engineering & Spatial CV

In [ ]:
# Spatial clusters for GroupKFold
stations = train_matched.groupby(['Latitude', 'Longitude']).size().reset_index().rename(columns={0: 'n'})
km = KMeans(n_clusters=5, random_state=42, n_init=10)
stations['spatial_cluster'] = km.fit_predict(stations[['Latitude', 'Longitude']])
train_matched = train_matched.merge(
    stations[['Latitude', 'Longitude', 'spatial_cluster']],
    on=['Latitude', 'Longitude'], how='left',
)

# Define feature columns (exclude targets, IDs, merge artifacts)
DROP_COLS = (
    TARGETS
    + ['Sample Date', 'spatial_cluster', 'geometry',
       '_merge_terra', '_merge_landsat',
       'Latitude_glorich', 'Longitude_glorich', 'date', 'dws_1st']
)
feature_cols = [c for c in train_matched.columns if c not in DROP_COLS]

dws_feats = [c for c in feature_cols if c.startswith('dws_')]
print(f"Total features: {len(feature_cols)}  (DWS: {len(dws_feats)})")

## Part 6: Train CatBoost with Spatial CV

In [ ]:
def evaluate_spatial_cv(X, y, groups, target_name, transform=None, n_splits=5):
    """Run GroupKFold spatial CV and report per-fold + OOF R²."""
    y_work = np.log1p(y) if transform == 'log1p' else y.copy()
    gkf = GroupKFold(n_splits=n_splits)

    cat_idx = [feature_cols.index(c) for c in ['Impute_Method'] if c in feature_cols]
    fold_scores = []
    oof_preds = np.full(len(y), np.nan)

    for fold_idx, (tr_idx, val_idx) in enumerate(gkf.split(X, groups=groups)):
        model = CatBoostRegressor(
            iterations=800, learning_rate=0.05, depth=6,
            l2_leaf_reg=3, verbose=0, random_state=42,
            cat_features=cat_idx or None,
        )
        model.fit(X.iloc[tr_idx], y_work.iloc[tr_idx])
        preds = model.predict(X.iloc[val_idx])

        if transform == 'log1p':
            preds_raw = np.expm1(preds)
            y_val_raw = y.iloc[val_idx]
        else:
            preds_raw, y_val_raw = preds, y_work.iloc[val_idx]

        score = r2_score(y_val_raw, preds_raw)
        fold_scores.append(score)
        oof_preds[val_idx] = preds_raw
        print(f"  Fold {fold_idx}: R²={score:.4f} (n={len(val_idx)})")

    valid = ~np.isnan(oof_preds)
    oof_r2 = r2_score(y[valid], oof_preds[valid])
    print(f"  → {target_name}: Mean R²={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}, OOF R²={oof_r2:.4f}")
    return {'mean_r2': np.mean(fold_scores), 'oof_r2': oof_r2}


X = train_matched[feature_cols].copy()
groups = train_matched['spatial_cluster'].values
results = {}

for target in TARGETS:
    y = train_matched[target].copy()
    transform = 'log1p' if target == 'Dissolved Reactive Phosphorus' else None
    print(f"\n{'='*50}")
    print(f"Target: {target} {'(log1p)' if transform else ''}")
    print(f"{'='*50}")
    results[target] = evaluate_spatial_cv(X, y, groups, target, transform)

mean_r2 = np.mean([r['mean_r2'] for r in results.values()])
print(f"\nCOMPETITION METRIC (mean R²): {mean_r2:.4f}")

## Part 7: Generate Hybrid Submission- **TAL**: DWS direct where available, CatBoost for gaps- **EC / DRP**: CatBoost (with DWS as features)

In [ ]:
X_train = train_matched[feature_cols].copy()
X_test = test_matched[feature_cols].copy()

cat_idx = [feature_cols.index(c) for c in ['Impute_Method'] if c in feature_cols]
test_preds = {}

for target in TARGETS:
    y = train_matched[target].copy()
    transform = 'log1p' if target == 'Dissolved Reactive Phosphorus' else None
    y_work = np.log1p(y) if transform == 'log1p' else y

    model = CatBoostRegressor(
        iterations=800, learning_rate=0.05, depth=6,
        l2_leaf_reg=3, verbose=100, random_state=42,
        cat_features=cat_idx or None,
    )
    model.fit(X_train, y_work)

    preds = model.predict(X_test)
    if transform == 'log1p':
        preds = np.expm1(preds)
    preds = np.clip(preds, 0, None)
    test_preds[target] = preds
    print(f"{target}: [{preds.min():.2f}, {preds.max():.2f}], mean={preds.mean():.2f}")

# Hybrid submission: TAL from DWS directly, EC/DRP from CatBoost
submission = pd.DataFrame({
    'Latitude': test_matched['Latitude'],
    'Longitude': test_matched['Longitude'],
    'Sample Date': test_matched['Sample Date'],
})

dws_tal = test_matched['dws_TAL'].values
submission['Total Alkalinity'] = np.where(np.isfinite(dws_tal), dws_tal, test_preds['Total Alkalinity'])
submission['Electrical Conductance'] = test_preds['Electrical Conductance']
submission['Dissolved Reactive Phosphorus'] = test_preds['Dissolved Reactive Phosphorus']

print(f"\nSubmission shape: {submission.shape}")
print(submission[TARGETS].describe())

submission.to_csv('submission_dws_hybrid.csv', index=False)
print("\nSaved: submission_dws_hybrid.csv")